In [1]:
%pip install langchain-huggingface dotenv scikit-learn requests sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [145]:
import numpy as np
from langchain_huggingface import HuggingFaceEmbeddings
from sklearn.metrics.pairwise import cosine_similarity
import requests
import os
from dotenv import load_dotenv
import re

load_dotenv()
ED_API_KEY = os.getenv("ED_API_KEY")

In [9]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

/Users/marcdavila/anaconda3/envs/myenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def fetch_threads(limit=50, sort="new"):
  """
  Fetches the Ed Thread API, guaraanteeing <limit> threads are returned. The 
  Ed API sets a hard limit of 100, so if more are needed, multiple requests
  are made with different offsets.
  TODO: handle rate limiting, fetch all threads if limit is None
  """
  threads = []
  while len(threads) < limit:
    res = requests.get(
      url=f"https://us.edstem.org/api/courses/74827/threads?limit={min(100, limit - len(threads))}&offset={len(threads)}&sort={sort}",
      headers={"Authorization": f"Bearer {ED_API_KEY}"}
    )
    data = res.json()
    threads.extend(data['threads'])
    if len(data['threads']) < 100:
      break
  threads = {
    thread["id"]: 
      {
        "title": thread["title"],
        "content": clean_xml_tags(thread["content"]).strip(),
        "title_embedding": embeddings.embed_query(thread["title"]),
        "content_embedding": embeddings.embed_query(clean_xml_tags(thread["content"]).strip())
  } for thread in threads}
  return threads

def getCommentsandAnswers(thread_ids):
    """
    Fetch all comments (including those on answers) for a list of thread IDs.
    Returns a list of comment texts.
    """
    def collect_all_comments(comment_list):
      """Recursively collect all comment texts from a list of comments."""
      texts = []
      for comment in comment_list:
          texts.append(comment.get('document', comment.get('content', '')))
          # Recursively collect nested comments
          if comment.get('comments'):
              texts.extend(collect_all_comments(comment['comments']))
      return texts
    

    all_comments = []
    all_answers = []
    # Goes through each thread id and gets the specific comments that it is
    # trying to get.
    for tid in thread_ids:
        res = requests.get(
            url=f"https://us.edstem.org/api/threads/{tid}",
            headers={"Authorization": f"Bearer {ED_API_KEY}"},
        )
        if res.status_code != 200:
            print(f"Failed to fetch thread {tid}: {res.status_code}")
            continue
        
        thread = res.json().get('thread', {})

        all_comments.extend(collect_all_comments(thread.get('comments', [])))
            
        for answer in thread.get('answers', []):
            all_answers.append(answer.get('document', answer.get('content', '')))
            all_comments.extend(collect_all_comments(answer.get('comments', [])))
    return all_comments, all_answers

def pre_processing(text):
  """Lowercase, remove punctuation, and extra whitespace."""
  text = text.lower()
  text = re.sub(r'[^\w\s]', '', text)
  text = re.sub(r'\s+', ' ', text)     
  return text.strip()


def search_threads(query, limit=20, sort="relevance", category=None, from_date=None, to_date=None):
  """
  Whole point of this function is to get the embeddings and the text of each 
  thread that matches the search query.

  """
  base = {
    "query": query,
    "limit": limit,
    "sort": sort,
    "category": category,
    "from_date": from_date,
    "to_date": to_date,
  }
  params = {k: v for k, v in base.items() if v is not None}
  res = requests.get(
    url="https://us.edstem.org/api/courses/74827/threads/search",
    headers={"Authorization": f"Bearer {ED_API_KEY}"},
    params=params
  )
  threads = res.json()['threads']
  result = {}
    
  for thread in threads:
    comments, answers = getCommentsandAnswers([thread["id"]])
    comment_texts = []
    answer_texts = []

    for comment in comments:
        comment_texts.append(clean_xml_tags(comment).strip())
    for answer in answers:
        answer_texts.append(clean_xml_tags(answer).strip())


    result[thread["id"]] = {
      "title": thread["title"],
      "content": pre_processing(clean_xml_tags(thread["title"]) + " " + 
                                clean_xml_tags(thread["content"]).strip() + " " + 
      " ".join(answer_texts) + " " + " ".join(comment_texts)),
    }

  for thread_id, data in result.items():
    # Generate title and content embeddings
    data["content_embedding"] = embeddings.embed_query(data["content"])

  return result

def clean_xml_tags(text):
  clean = re.compile('<.*?>')
  return re.sub(clean, '', text)


In [149]:
text = search_threads('diffusion math',
                limit=5,
                sort='relevance')

print(text)

{6436163: {'title': 'posterior and prior', 'content': 'posterior and prior just to clarify in elbo for diffusion and vae what is the posterior and what is the prior for vaes the prior is the distribution of our latent space which we represent with pz the assumption is that this is normally distributed the posterior is the distribution that our encoder approximates qz x in general the prior refers to the probability of your hypothesis the latent space value z before seeing any data the posterior is the revised probability after observing data the data point x for diffusion models it gets more complicated and im not confident enough to answer the idea is similar but the denoising process and time sampling make it harder i think posterior and prior are more general terms looking at this try to understand which part of diffusion is the prior and which is the posterior', 'content_embedding': [0.013128668069839478, -0.019864987581968307, 0.011099652387201786, -0.0411047488451004, 0.011221815

In [150]:
def find_top_k_results(q, threads, k=5):
  """
  Find the top k results from thread content based on cosine similarity to query q.
  Returns a list of (thread_id, similarity) tuples.
  """

  results = sorted(((item[0], cosine_similarity([q], [item[1]['content_embedding']])[0][0]) for item in threads.items()), key=lambda item: item[1], reverse=True)
  for thread_id, sim in results[:k]:
    print(f"Thread ID: {thread_id}, Similarity: {sim:.4f}")
    print(f"Title: {threads[thread_id]['title']}")
    print(f"Content: {threads[thread_id]['content'][:200]}...")
    print()
  return results[:k]

In [151]:
all_threads = fetch_threads(limit=600, sort="new")
len(all_threads)

584

In [152]:
q = "Can someone explain transformers"
results = find_top_k_results(embeddings.embed_query(q), all_threads, k=5)

Thread ID: 6332379, Similarity: 0.4605
Title: Why are Transformers so much faster than e.g. LSTMs?
Content: One thing some students still seem confused about is why transformers are so much more popular, if they are still autoregressive (i.e. they are generating token after token left to right). The answer ...

Thread ID: 6436211, Similarity: 0.4206
Title: question abt CLIP/swin transformers
Content: are CLIP and Swin transformers considered self-supervised or supervised learning?...

Thread ID: 6332365, Similarity: 0.3684
Title: Transformer Accuracy Right, but number of layers not
Content: My autograder says the number of layers is not right for the transformer architecture but somehow the accuracy is, how can this be ? Is this just pure luck...

Thread ID: 6432462, Similarity: 0.3252
Title: Practice midterm q2 8)
Content: Transforms samples from a unit normal distribution to samples from the data distribution.Does this apply to all transformers not just the image ones?...

Thread ID:

In [153]:
search = search_threads(q, limit=5)
search_sim = {thread_id: cosine_similarity([embeddings.embed_query(q)], [data['content_embedding']])[0][0] for thread_id, data in search.items()}
for thread_id, sim in search_sim.items():
  print(f"Thread ID: {thread_id}, Similarity: {sim:.4f}")
  print(f"Title: {search[thread_id]['title']}")
  print(f"Content: {search[thread_id]['content'][:200]}...")
  print()

Thread ID: 6436211, Similarity: 0.4238
Title: question abt CLIP/swin transformers
Content: question abt clipswin transformers are clip and swin transformers considered selfsupervised or supervised learning clip isnt a transformer in and of itself but it uses encoders which are often transfo...

Thread ID: 6432462, Similarity: 0.4021
Title: Practice midterm q2 8)
Content: practice midterm q2 8 transforms samples from a unit normal distribution to samples from the data distributiondoes this apply to all transformers not just the image ones no since nlp transformers for ...

Thread ID: 6332379, Similarity: 0.4164
Title: Why are Transformers so much faster than e.g. LSTMs?
Content: why are transformers so much faster than eg lstms one thing some students still seem confused about is why transformers are so much more popular if they are still autoregressive ie they are generating...

Thread ID: 6577439, Similarity: 0.1639
Title: ALiBi (Attention with Linear Biases)
Content: alibi attention 